In [ ]:
%pip install -q "transformers==4.53.3" "datasets==3.6.0" "accelerate==1.8.1" "torch>=2.6" scikit-learn

In [ ]:
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

dataset_id = "argilla/synthetic-domain-text-classification"
model_id = "answerdotai/ModernBERT-base"
output_dir = Path("modernbert-domain-classifier")
seed = 42
max_length = 512

set_seed(seed)

In [ ]:
dataset = load_dataset(dataset_id, split="train")
labels = dataset.features["label"].names
id2label = dict(enumerate(labels))
label2id = {name: index for index, name in id2label.items()}

split = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column="label")
holdout = split["test"].train_test_split(test_size=0.5, seed=seed, stratify_by_column="label")
train = split["train"]
validation = holdout["train"]
test = holdout["test"]

print("Кількість класів:", len(labels))
print("Кількість прикладів у train, validation і test:", len(train), len(validation), len(test))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

def prepare_split(split):
    return split.rename_column("label", "labels").map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
    )

train_tokens = prepare_split(train)
validation_tokens = prepare_split(validation)
test_tokens = prepare_split(test)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    attn_implementation="sdpa",
)

def compute_metrics(eval_prediction):
    logits, targets = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(targets, predictions),
        "f1_weighted": f1_score(targets, predictions, average="weighted", zero_division=0),
        "f1_macro": f1_score(targets, predictions, average="macro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=validation_tokens,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_tokens, metric_key_prefix="test")
print("Test accuracy:", round(test_metrics["test_accuracy"], 4))
print("Test weighted F1:", round(test_metrics["test_f1_weighted"], 4))
print("Test macro F1:", round(test_metrics["test_f1_macro"], 4))

trainer.save_model(str(output_dir))

In [ ]:
classifier = pipeline(
    "text-classification",
    model=str(output_dir),
    tokenizer=str(output_dir),
    device=training_args.device,
)

examples = [
    "The central bank raised interest rates to slow inflation.",
    "The bakery sells sourdough bread and pastries every morning.",
    "The football team won after scoring twice in the second half.",
]

for sample, prediction in zip(examples, classifier(examples, truncation=True, max_length=max_length)):
    print(sample, prediction["label"], round(prediction["score"], 4))